# 🎓 Student Performance Prediction - Exploratory Data Analysis & Modeling
This notebook provides a complete walk-through of the Student Performance Prediction project:
1. Data Loading & Inspection
2. Data Preprocessing & Missing Value Imputation
3. Encoding Categorical Features & Scaling
4. Exploratory Data Analysis (EDA) & Correlation Heatmap
5. Multi-Model Benchmark (Classification & Regression)
6. Hyperparameter Tuning & Cross Validation
7. Model Evaluation (Confusion Matrix, ROC Curve, Feature Importance)

In [ ]:
# Step 1: Import Required Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBClassifier, XGBRegressor

%matplotlib inline
sns.set_style('whitegrid')

In [ ]:
# Step 2: Load Dataset
dataset_path = '../dataset/student_performance.csv'
df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Step 3: Dataset Summary & Missing Values Inspection
print("--- Data Information ---")
print(df.info())
print("
--- Missing Values Count ---")
print(df.isnull().sum())
print("
--- Summary Statistics ---")
df.describe().T

In [ ]:
# Step 4: Handle Missing Values
numeric_cols = ['Hours Studied', 'Attendance (%)', 'Previous Exam Score', 'Assignment Score', 'Sleep Hours', 'Study Time', 'Age']
categorical_cols = ['Internet Access', 'Family Income', 'Parent Education', 'Extra Activities', 'Gender']

num_imputer = SimpleImputer(strategy='median')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

print("Missing values after imputation:", df.isnull().sum().sum())

In [ ]:
# Step 5: Label & One-Hot Encoding
df['Target_Class'] = (df['Final Exam Score'] >= 70).astype(int)

parent_edu_order = {'High School': 0, 'Bachelor': 1, 'Master': 2, 'Doctorate': 3}
family_income_order = {'Low': 0, 'Medium': 1, 'High': 2}

df_encoded = df.copy()
df_encoded['Parent Education'] = df_encoded['Parent Education'].map(parent_edu_order)
df_encoded['Family Income'] = df_encoded['Family Income'].map(family_income_order)

nominal_cols = ['Internet Access', 'Extra Activities', 'Gender']
ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe_feat = ohe.fit_transform(df_encoded[nominal_cols])
ohe_df = pd.DataFrame(ohe_feat, columns=ohe.get_feature_names_out(nominal_cols))

df_final = pd.concat([df_encoded[numeric_cols + ['Parent Education', 'Family Income']], ohe_df, df_encoded[['Final Exam Score', 'Target_Class']]], axis=1)
df_final.head()

In [ ]:
# Step 6: Exploratory Data Analysis & Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df_final.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Step 7: Feature Scaling & Train-Test Split
feature_cols = [c for c in df_final.columns if c not in ['Final Exam Score', 'Target_Class']]
X = df_final[feature_cols]
y_cls = df_final['Target_Class']
y_reg = df_final['Final Exam Score']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

X_train, X_test, y_train_cls, y_test_cls = train_test_split(X_scaled, y_cls, test_size=0.2, random_state=42, stratify=y_cls)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

In [ ]:
# Step 8: Train & Benchmark Classification Models
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train_cls)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else preds
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test_cls, preds),
        'Precision': precision_score(y_test_cls, preds),
        'Recall': recall_score(y_test_cls, preds),
        'F1 Score': f1_score(y_test_cls, preds),
        'ROC AUC': roc_auc_score(y_test_cls, probs)
    })

pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)